## 1  Imports

In [1]:
import matplotlib.dates as mdates
import datetime
import matplotlib.pyplot as plt
import numpy as np
import os
import scipy.ndimage
import DetectRadioburst_OSRA as drb

from DetectRadioburst_OSRA import append_daily_csv

from astropy.time import Time
from astropy.coordinates import EarthLocation, get_sun, AltAz
import astropy.units as u

## 2  Load data

In [2]:
# ── File path — first configurable line ──────────────────────────────────────
fname = "/net/lyot/scratch3/vocks/OSRA/2003/CD_300/031027_300.roh"

# Load full CD (day + night)
dyspec, t_fits, f_fits = drb.read_osraf2(fname)
t_fits = drb.to_python_datetime(t_fits)   #time conversion

print(f"Loaded {len(t_fits)} records")
print(f"Time range : {t_fits[0]}  ->  {t_fits[-1]}")

Loaded 594606 records
Time range : 2003-10-27 03:00:02  ->  2003-10-27 19:31:02.800000


## 3  Sunrise / Sunset — Tremsdorf
> **From v2:** sparse sampling — `compute_sunrise_sunset()` called on ~100 points instead of every record.

In [3]:
# ── Tremsdorf observatory coordinates ─────────────────────────────────────────
TREMSDORF = EarthLocation(
    lat=52.2167 * u.deg,
    lon=13.1833 * u.deg,
    height=65 * u.m
)

def compute_sunrise_sunset(t_fits, location=TREMSDORF, step=None):
    """
    Compute sunrise and sunset times from the observation time array.

    Uses sparse sampling for speed — 100 evenly spaced points from t_fits
    are enough to locate the zero-crossing of solar altitude.

    Parameters
    ----------
    t_fits   : 1D array of Python datetime objects (full day)
    location : astropy EarthLocation
    step     : sampling interval — computed automatically if None

    Returns
    -------
    sunrise_str : str  "HH:MM" UT  or  "NO_RISE" if sun never rises
    sunset_str  : str  "HH:MM" UT  or  "NO_SET"  if sun never sets
    sunrise_dt  : datetime object  (for mask use)
    sunset_dt   : datetime object  (for mask use)
    """

    # Sparse sampling — 100 points is enough to locate the crossing
    if step is None:
        step = max(1, len(t_fits) // 100)
    t_sparse = t_fits[::step]

    times       = Time(list(t_sparse))
    altaz_frame = AltAz(obstime=times, location=location)
    sun_altaz   = get_sun(times).transform_to(altaz_frame)

    # +0.833 deg is the standard atmospheric refraction correction
    # that defines the official sunrise/sunset moment
    altitudes   = sun_altaz.alt.deg + 0.833

    sunrise_idx = np.where((altitudes[:-1] < 0) & (altitudes[1:] >= 0))[0]
    sunset_idx  = np.where((altitudes[:-1] >= 0) & (altitudes[1:] < 0))[0]

    if len(sunrise_idx) > 0:
        sunrise_dt  = t_sparse[sunrise_idx[0]]
        sunrise_str = sunrise_dt.strftime("%H:%M")
    else:
        sunrise_dt  = t_fits[0]    # fallback: start of file
        sunrise_str = "NO_RISE"

    if len(sunset_idx) > 0:
        sunset_dt  = t_sparse[sunset_idx[0]]
        sunset_str = sunset_dt.strftime("%H:%M")
    else:
        sunset_dt  = t_fits[-1]    # fallback: end of file
        sunset_str = "NO_SET"

    return sunrise_str, sunset_str, sunrise_dt, sunset_dt

In [4]:
# ── Compute once before the loop ─────────────────────────────────────────────
sunrise_str, sunset_str, sunrise_dt, sunset_dt = compute_sunrise_sunset(t_fits)

print(f"Sunrise : {sunrise_str} UT")
print(f"Sunset  : {sunset_str}  UT")

# ── Initialise the results list before the loop ───────────────────────────────
# CRITICAL: this must be outside the loop.
# If it were inside, it would reset to [] on every iteration and you
# would only ever have one hour's data when you call append_daily_csv.
hourly_results = []

# ── Hour loop — all 24 hours ──────────────────────────────────────────────────
for h in range(24):

    start_time = datetime.datetime(
        t_fits[0].year, t_fits[0].month, t_fits[0].day, h, 0, 0
    )
    end_time = start_time + datetime.timedelta(hours=1)

    time_mask = (t_fits >= start_time) & (t_fits < end_time)
    idx       = np.where(time_mask)[0]

    # ── Daylight annotation ───────────────────────────────────────────────────
    hour_midpoint = start_time + datetime.timedelta(minutes=30)
    daylight_flag = "DAY" if (sunrise_dt <= hour_midpoint <= sunset_dt) else "NIGHT"

    # ── CASE 1: no data in this hour ──────────────────────────────────────────
    # The file may not cover the full 24 hours — e.g. observation started
    # at 06:00 UT, so hours 00–05 are empty.
    # We still record the hour so the CSV has a complete 24-row block
    # every day — no gaps, no missing rows.
    if len(idx) == 0:
        hourly_results.append({
            "hour"       : h,
            "bursts"     : 0,
            "raw_groups" : 0,
            "samples"    : 0,
            "daylight"   : daylight_flag,
            "sunrise_ut" : sunrise_str,
            "sunset_ut"  : sunset_str
        })
        print(f"Hour {h:02d}: no data")
        continue   # move to next hour

    # ── Cut data for this hour ────────────────────────────────────────────────
    data_cut   = dyspec[idx, :]
    t_cut      = t_fits[idx]

    # ── Preprocess ───────────────────────────────────────────────────────────
    data_fits_new, data_fits_new_smooth = drb.preproc2(
        data_cut, gauss_sigma=(5.5, 0.0)
    )

    # ── CASE 2: preproc2 returned all-zero smooth array ───────────────────────
    # Happens when the hour contains only dead or saturated channels.
    # binarization on a zero array produces meaningless output — record
    # zeros and skip to the next hour rather than polluting the CSV.
    if np.all(data_fits_new_smooth == 0):
        hourly_results.append({
            "hour"       : h,
            "bursts"     : 0,
            "raw_groups" : 0,
            "samples"    : len(idx),
            "daylight"   : daylight_flag,
            "sunrise_ut" : sunrise_str,
            "sunset_ut"  : sunset_str
        })
        print(f"Hour {h:02d}: blank spectrum after preproc")
        continue

    # ── Binarization ─────────────────────────────────────────────────────────
    bmap = drb.binarization(data_fits_new_smooth, N_order=8, peak_r=0.9993)

    # ── Hough detection ───────────────────────────────────────────────────────
    lines = drb.hough_detect(
        bmap, data_cut,
        threshold=30, line_gap=10, line_length=50,
        theta=np.linspace(np.pi/2 - np.pi/8, np.pi/2 - 0.5/180*np.pi, 300)
    )

    # ── CASE 3: no Hough lines found ──────────────────────────────────────────
    # Quiet hour — no features detected. Record zeros but keep the samples
    # count so you can distinguish a truly quiet hour from a data gap.
    if len(lines) == 0:
        hourly_results.append({
            "hour"       : h,
            "bursts"     : 0,
            "raw_groups" : 0,
            "samples"    : len(idx),
            "daylight"   : daylight_flag,
            "sunrise_ut" : sunrise_str,
            "sunset_ut"  : sunset_str
        })
        print(f"Hour {h:02d}: no Hough lines  (samples={len(idx)})")
        continue

    # ── Group lines ───────────────────────────────────────────────────────────
    # line_grouping returns [] for empty input (already edited in drb).
    line_sets = drb.line_grouping(lines, min_dist=10)

    # ── CASE 4: grouping returned no groups ───────────────────────────────────
    # Extremely rare but possible if all lines were degenerate.
    if len(line_sets) == 0:
        hourly_results.append({
            "hour"       : h,
            "bursts"     : 0,
            "raw_groups" : 0,
            "samples"    : len(idx),
            "daylight"   : daylight_flag,
            "sunrise_ut" : sunrise_str,
            "sunset_ut"  : sunset_str
        })
        print(f"Hour {h:02d}: lines found but no groups formed  (samples={len(idx)})")
        continue

    # ── Count bursts ──────────────────────────────────────────────────────────
    # confirmed: groups with more than 1 Hough line — real drift tracks
    # raw_groups: all groups including single-line noise candidates
    confirmed  = sum(1 for ls in line_sets if len(ls) > 1)
    raw_groups = len(line_sets)

    # ── GUARANTEED APPEND — reached only when detection ran fully ────────────
    # All four edge cases above used continue, so if execution reaches
    # here it means: data existed, preproc ran, lines were found,
    # groups were formed. This append will always execute for active hours.
    hourly_results.append({
        "hour"       : h,
        "bursts"     : confirmed,
        "raw_groups" : raw_groups,
        "samples"    : len(idx),
        "daylight"   : daylight_flag,
        "sunrise_ut" : sunrise_str,
        "sunset_ut"  : sunset_str
    })

    print(f"Hour {h:02d}: bursts={confirmed:>3}  "
          f"raw_groups={raw_groups:>3}  "
          f"samples={len(idx):>6}  "
          f"[{daylight_flag}]")

# ── SAFETY CHECK before saving ────────────────────────────────────────────────
# Verify the list has exactly 24 entries — one per hour, no gaps.
# If this assertion fails it means a continue was reached without
# an append, which indicates a bug in the loop above.
assert len(hourly_results) == 24, (
    f"Expected 24 hourly records, got {len(hourly_results)}. "
    f"Check that every branch of the hour loop calls hourly_results.append()."
)

# ── Save to CSV — called ONCE after the loop, never inside it ────────────────
append_daily_csv(
    hourly_results = hourly_results,
    date           = t_fits[0].strftime("%Y-%m-%d"),   # auto-extract from data
    output_dir     = "outputs/solar_cycle"
)

Sunrise : 05:48 UT
Sunset  : 15:43  UT
Hour 00: no data
Hour 01: no data
Hour 02: no data
Hour 03: bursts=  1  raw_groups=  1  samples= 35974  [NIGHT]
Hour 04: bursts=  0  raw_groups=  1  samples= 36000  [NIGHT]
Hour 05: bursts=  0  raw_groups=  1  samples= 36000  [NIGHT]
Hour 06: bursts=  1  raw_groups=  2  samples= 36002  [DAY]
Hour 07: bursts=725  raw_groups=958  samples= 35996  [DAY]
Hour 08: bursts= 90  raw_groups=151  samples= 36002  [DAY]
Hour 09: bursts=  0  raw_groups=  3  samples= 36005  [DAY]
Hour 10: bursts=  0  raw_groups=  4  samples= 35991  [DAY]
Hour 11: bursts=  1  raw_groups=  7  samples= 36005  [DAY]
Hour 12: bursts= 17  raw_groups= 62  samples= 36002  [DAY]
Hour 13: bursts= 10  raw_groups= 44  samples= 35995  [DAY]
Hour 14: bursts=230  raw_groups=377  samples= 36002  [DAY]
Hour 15: bursts= 18  raw_groups= 42  samples= 35998  [DAY]
Hour 16: bursts=  0  raw_groups=  1  samples= 36001  [NIGHT]
Hour 17: bursts=  1  raw_groups=  1  samples= 36005  [NIGHT]
Hour 18: bursts

'outputs/solar_cycle/bursts_2003.csv'